# Lab 05: Cryptanalysis & Flaws in Real-World RSA Deployments
**Student Name:** Md. Akib Al Jawad  
**Student ID:** IT-23020  
**Department:** Information and Communication Technology (ICT)  
**Institution:** Mawlana Bhashani Science and Technology University (MBSTU)  
**Reference:** IIT Bombay Maths and Physics Club Workshop — RSA Assignment Notebook

---

## Executive Summary
Textbook RSA is deterministic: $c = m^e \bmod N$. Without probabilistic padding schemes (such as OAEP - Optimal Asymmetric Encryption Padding) and careful key generation parameters, RSA is susceptible to severe structural vulnerabilities. This notebook solves all 7 challenge questions from the official assignment:
1. **Basic Encryption**: Function `Encrypt(message, modulo, exponent)`.
2. **Basic Decryption**: Function `Decrypt(ciphertext, p, q, exponent)`.
3. **Attack 1 (Dictionary / Small Message Space)**: Exploiting deterministic encryption when plaintext is chosen from a small candidate pool.
4. **Attack 2 (Small Prime Factorization)**: Trial division attack when $\min(p, q) \le 1,000,000$.
5. **Attack 3 (Fermat Factorization)**: Factoring modulus when primes are too close: $|p - q| < 5000$.
6. **Attack 4 (Batch GCD / Shared Modulus Attack)**: Breaking independent keys that share a prime factor due to entropy failure: $\gcd(N_1, N_2) = p$.
7. **Attack 5 (Håstad's Broadcast Attack)**: Eavesdropping identical messages sent to multiple recipients with small exponent $e=2$ using CRT and integer root.



In [1]:
from scripts.rsa_attacks import (
    ConvertToInt, ConvertToStr, PowMod, ExtendedEuclid, InvertModulo,
    Encrypt, Decrypt, DecipherSimple, DecipherSmallPrime, DecipherSmallDiff,
    GCD, DecipherCommonDivisor, IntSqrt, ChineseRemainderTheorem, DecipherHastad
)

print("[*] All Cryptanalytic Attack Primitives Loaded Successfully.")



[*] All Cryptanalytic Attack Primitives Loaded Successfully.


### Questions 1 & 2: RSA Encryption and Decryption Primitives
- `Encrypt(message, modulo, exponent)` computes $m^e \bmod N$.
- `Decrypt(ciphertext, p, q, exponent)` computes $d = e^{-1} \bmod ((p-1)(q-1))$, then recovers $m = c^d \bmod (pq)$.



In [1]:
p = 1000000007
q = 1000000009
exponent = 23917
modulo = p * q

ciphertext = Encrypt("attack", modulo, exponent)
recovered = Decrypt(ciphertext, p, q, exponent)
print(f"Ciphertext: {ciphertext}")
print(f"Decrypted:  '{recovered}'")
assert recovered == "attack"



Ciphertext: 577821702011958447


Decrypted:  'attack'


### Question 3: Attack 1 — Small Message Space Dictionary Attack
If an adversary intercepts ciphertext $c$ and knows the candidate message is one of `["attack", "don't attack", "wait"]`, they can simply encrypt all candidate messages with Alice's public key until finding a match.



In [1]:
modulo = 101
exponent = 12
candidates = ["attack", "don't attack", "wait"]
target_cipher = Encrypt("attack", modulo, exponent)

deciphered = DecipherSimple(target_cipher, modulo, exponent, candidates)
print(f"Target Ciphertext: {target_cipher}")
print(f"Deciphered Result: '{deciphered}'")
assert deciphered == "attack"



Target Ciphertext: 78


Deciphered Result: 'attack'


### Question 4: Attack 2 — Factoring with a Small Prime Factor ($p < 1,000,000$)
If one prime factor is small ($p < 10^6$), standard trial division finds $p$ in sub-second time regardless of the size of the other prime factor $q$.



In [1]:
small_p = 101
large_q = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
modulo = small_p * large_q
exponent = 239

ct_sp = Encrypt("attack", modulo, exponent)
recovered_sp = DecipherSmallPrime(ct_sp, modulo, exponent)
print(f"Deciphered Small Prime Challenge: '{recovered_sp}'")
assert recovered_sp == "attack"



Deciphered Small Prime Challenge: 'attack'


### Question 5: Attack 3 — Fermat's Factorization for Close Primes ($|p - q| < 5000$)
Let $N = pq$ with $p > q$. Let $a = \frac{p + q}{2}$ and $b = \frac{p - q}{2}$.  
Then:
$$a^2 - b^2 = \left(\frac{p+q}{2}\right)^2 - \left(\frac{p-q}{2}\right)^2 = pq = N$$
If $|p - q| = 2b < 5000$, then $b < 2500$ and $b^2 < 6.25 \times 10^6$.  
Hence $a = \sqrt{N + b^2} \approx \sqrt{N}$. Searching $a$ beginning from $\lceil \sqrt{N} \rceil$ finds $b = \sqrt{a^2 - N}$ in very few iterations!



In [1]:
p_diff = 1000000007
q_diff = 1000000009
n_diff = p_diff * q_diff
e_diff = 239

ct_sd = Encrypt("attack", n_diff, e_diff)
recovered_sd = DecipherSmallDiff(ct_sd, n_diff, e_diff)
print(f"Deciphered Small Difference Challenge: '{recovered_sd}'")
assert recovered_sd == "attack"



Deciphered Small Difference Challenge: 'attack'


### Question 6: Attack 4 — Shared Prime Factor (Batch GCD Attack)
When two systems instantiate RSA keys using flawed pseudo-random generators, they may independently select the same prime $p$.  
Given $N_1 = p \cdot q_1$ and $N_2 = p \cdot q_2$:
$$\gcd(N_1, N_2) = p$$
Euclidean GCD takes negligible time, instantly yielding $p$, $q_1 = N_1 / p$, and $q_2 = N_2 / p$.



In [1]:
p_shared = 101
n1 = p_shared * large_q
n2 = p_shared * 1000000007
e1 = 239
e2 = 17

c1 = Encrypt("attack", n1, e1)
c2 = Encrypt("wait", n2, e2)

m1, m2 = DecipherCommonDivisor(c1, n1, e1, c2, n2, e2)
print(f"Deciphered Alice:    '{m1}'")
print(f"Deciphered Angelina: '{m2}'")
assert m1 == "attack" and m2 == "wait"



Deciphered Alice:    'attack'


Deciphered Angelina: 'wait'


### Question 7: Attack 5 — Håstad's Broadcast Attack ($e = 2$)
Suppose the same message $m$ is encrypted and broadcast to two recipients with coprimes moduli $N_1, N_2$ with $e = 2$:
$$c_1 \equiv m^2 \pmod{N_1}, \quad c_2 \equiv m^2 \pmod{N_2}$$
By the Chinese Remainder Theorem, we find:
$$c \equiv m^2 \pmod{N_1 N_2}$$
Since $m < N_1$ and $m < N_2$, it follows that $m^2 < N_1 N_2$.  
Therefore, modular reduction **never wrapped around**! The integer $c$ is the exact mathematical square $m^2$ over $\mathbb{Z}$. Taking the exact integer square root $m = \lfloor \sqrt{c} \rfloor$ completely recovers $m$ without factoring $N_1$ or $N_2$.



In [1]:
p1_h = 790383132652258876190399065097
q1_h = 662503581792812531719955475509
p2_h = 656917682542437675078478868539
q2_h = 1263581691331332127259083713503
n1_h = p1_h * q1_h
n2_h = p2_h * q2_h

c1_h = Encrypt("attack", n1_h, 2)
c2_h = Encrypt("attack", n2_h, 2)

recovered_h = DecipherHastad(c1_h, n1_h, c2_h, n2_h)
print(f"Deciphered Broadcast Message: '{recovered_h}'")
assert recovered_h == "attack"



Deciphered Broadcast Message: 'attack'
